In [3]:
import pandas as pd

In [6]:
df = pd.read_csv("nyc_311_sample.csv")
df.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,status,borough,incident_zip
0,63950044,2025-01-31T23:59:48.000,2025-02-01T04:44:27.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Closed,MANHATTAN,10033.0
1,63945541,2025-01-31T23:59:45.000,2025-02-01T17:35:57.000,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,Closed,BROOKLYN,11230.0
2,63947478,2025-01-31T23:59:19.000,2025-02-01T10:33:54.000,NYPD,New York City Police Department,Blocked Driveway,Partial Access,Closed,BROOKLYN,11229.0
3,63950000,2025-01-31T23:59:00.000,2025-02-03T23:00:00.000,DEP,Department of Environmental Protection,Noise,Noise: Construction Before/After Hours (NM1),Closed,BRONX,10459.0
4,63948032,2025-01-31T23:59:00.000,2025-02-01T06:55:00.000,DOT,Department of Transportation,Traffic Signal Condition,Controller,Closed,MANHATTAN,10004.0


In [7]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (300, 10)
Columns: ['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'status', 'borough', 'incident_zip']


In [8]:
display(df.isna().sum().to_frame("missing_values"))

,missing_values
unique_key,0
created_date,0
closed_date,0
agency,0
agency_name,0
complaint_type,0
descriptor,1
status,0
borough,0
incident_zip,3


In [9]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate complaint IDs:", df["unique_key"].duplicated().sum())
display(df["status"].value_counts(dropna=False).to_frame("count"))

Exact duplicate rows: 0
Duplicate complaint IDs: 0


,count
status,
Closed,300


### Clean the data

In [10]:
clean = df.copy()

# Trim text values and treat empty strings as missing.
text_columns = [
    "unique_key", "agency", "agency_name", "complaint_type",
    "descriptor", "status", "borough", "incident_zip"
]
for col in text_columns:
    clean[col] = clean[col].astype("string").str.strip()
    clean[col] = clean[col].replace("", pd.NA)

# Convert date columns to datetime; invalid values become NaT.
clean["created_date"] = pd.to_datetime(clean["created_date"], errors="coerce")
clean["closed_date"] = pd.to_datetime(clean["closed_date"], errors="coerce")

rows_before = len(clean)
clean = clean.dropna(subset=["unique_key", "created_date"])
clean = clean.drop_duplicates(subset=["unique_key"], keep="first")

# Fill missing descriptive values, but preserve missing closed dates and ZIP codes.
for col in ["agency", "agency_name", "complaint_type", "descriptor", "status", "borough"]:
    clean[col] = clean[col].fillna("Unknown")

clean["complaint_id"] = pd.to_numeric(clean["unique_key"], errors="coerce")
clean = clean.dropna(subset=["complaint_id"])
clean["complaint_id"] = clean["complaint_id"].astype("int64")

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(clean))
print("Rows removed:", rows_before - len(clean))
print("Created date range:", clean["created_date"].min(), "to", clean["created_date"].max())
print("Closed before created:", int((clean["closed_date"].notna() & (clean["closed_date"] < clean["created_date"])).sum()))
display(clean.head())

Rows before cleaning: 300
Rows after cleaning: 300
Rows removed: 0
Created date range: 2025-01-31 23:13:00 to 2025-01-31 23:59:48
Closed before created: 0


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,status,borough,incident_zip,complaint_id
0,63950044,2025-01-31 23:59:48,2025-02-01 04:44:27,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Closed,MANHATTAN,10033.0,63950044
1,63945541,2025-01-31 23:59:45,2025-02-01 17:35:57,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,Closed,BROOKLYN,11230.0,63945541
2,63947478,2025-01-31 23:59:19,2025-02-01 10:33:54,NYPD,New York City Police Department,Blocked Driveway,Partial Access,Closed,BROOKLYN,11229.0,63947478
3,63950000,2025-01-31 23:59:00,2025-02-03 23:00:00,DEP,Department of Environmental Protection,Noise,Noise: Construction Before/After Hours (NM1),Closed,BRONX,10459.0,63950000
4,63948032,2025-01-31 23:59:00,2025-02-01 06:55:00,DOT,Department of Transportation,Traffic Signal Condition,Controller,Closed,MANHATTAN,10004.0,63948032


### Prepare the agency and category tables

In [11]:
agencies_df = (
    clean[["agency", "agency_name"]]
    .drop_duplicates()
    .rename(columns={"agency": "agency_code"})
    .sort_values(["agency_code", "agency_name"])
    .reset_index(drop=True)
)

categories_df = (
    clean[["complaint_type", "descriptor"]]
    .drop_duplicates()
    .sort_values(["complaint_type", "descriptor"])
    .reset_index(drop=True)
)

print("Distinct agencies:", len(agencies_df))
print("Distinct complaint categories:", len(categories_df))
display(agencies_df.head())
display(categories_df.head())

Distinct agencies: 9
Distinct complaint categories: 55


,agency_code,agency_name
0,DCWP,Department of Consumer and Worker Protection
1,DEP,Department of Environmental Protection
2,DOB,Department of Buildings
3,DOHMH,Department of Health and Mental Hygiene
4,DOT,Department of Transportation


,complaint_type,descriptor
0,Abandoned Vehicle,With License Plate
1,Air Quality,"Air: Odor/Fumes, Restaurant (AD2)"
2,Animal-Abuse,Neglected
3,Blocked Driveway,No Access
4,Blocked Driveway,Partial Access


### Connect Jupyter to MySQL

In [12]:
from sqlalchemy import create_engine, URL, text
from getpass import getpass

MYSQL_USER = input("MySQL username (often root): ").strip()
MYSQL_PASSWORD = getpass("MySQL password: ")

connection_url = URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host="localhost",
    port=3306,
    database="civic_complaint_intelligence",
)
engine = create_engine(connection_url)

with engine.connect() as connection:
    print("MySQL version:", connection.execute(text("SELECT VERSION()")).scalar())
    print("Connected successfully.")

MySQL username (often root):  root
MySQL password:  ········


MySQL version: 8.0.46
Connected successfully.


### Create the three relational tables

In [13]:
create_tables_sql = [
    '''
    CREATE TABLE IF NOT EXISTS agencies (
        agency_id INT AUTO_INCREMENT PRIMARY KEY,
        agency_code VARCHAR(50) NOT NULL,
        agency_name VARCHAR(255) NOT NULL,
        UNIQUE KEY uq_agency (agency_code, agency_name)
    )
    ''',
    '''
    CREATE TABLE IF NOT EXISTS complaint_categories (
        category_id INT AUTO_INCREMENT PRIMARY KEY,
        complaint_type VARCHAR(255) NOT NULL,
        descriptor VARCHAR(500) NOT NULL,
        UNIQUE KEY uq_category (complaint_type, descriptor)
    )
    ''',
    '''
    CREATE TABLE IF NOT EXISTS complaints (
        complaint_id BIGINT PRIMARY KEY,
        created_date DATETIME NOT NULL,
        closed_date DATETIME NULL,
        agency_id INT NOT NULL,
        category_id INT NOT NULL,
        status VARCHAR(100) NOT NULL,
        borough VARCHAR(100) NOT NULL,
        incident_zip VARCHAR(20) NULL,
        CONSTRAINT fk_complaints_agency FOREIGN KEY (agency_id) REFERENCES agencies(agency_id),
        CONSTRAINT fk_complaints_category FOREIGN KEY (category_id) REFERENCES complaint_categories(category_id),
        INDEX idx_created_date (created_date),
        INDEX idx_closed_date (closed_date),
        INDEX idx_status (status),
        INDEX idx_borough (borough)
    )
    '''
]

with engine.begin() as connection:
    for statement in create_tables_sql:
        connection.execute(text(statement))

print("Tables created or already exist.")

Tables created or already exist.


### Upload the cleaned data

In [14]:
# Delete child records before parent records to respect foreign keys.
with engine.begin() as connection:
    connection.execute(text("DELETE FROM complaints"))
    connection.execute(text("DELETE FROM complaint_categories"))
    connection.execute(text("DELETE FROM agencies"))

# Upload dimension tables first; MySQL generates their IDs.
agencies_df.to_sql("agencies", con=engine, if_exists="append", index=False, method="multi", chunksize=500)
categories_df.to_sql("complaint_categories", con=engine, if_exists="append", index=False, method="multi", chunksize=500)

agency_map = pd.read_sql("SELECT agency_id, agency_code, agency_name FROM agencies", con=engine)
category_map = pd.read_sql("SELECT category_id, complaint_type, descriptor FROM complaint_categories", con=engine)

upload = clean[[
    "complaint_id", "created_date", "closed_date", "agency", "agency_name",
    "complaint_type", "descriptor", "status", "borough", "incident_zip"
]].copy()

upload = upload.merge(
    agency_map, left_on=["agency", "agency_name"],
    right_on=["agency_code", "agency_name"], how="left", validate="many_to_one"
)
upload = upload.merge(
    category_map, on=["complaint_type", "descriptor"],
    how="left", validate="many_to_one"
)

upload = upload[[
    "complaint_id", "created_date", "closed_date", "agency_id",
    "category_id", "status", "borough", "incident_zip"
]]
upload = upload.astype(object).where(pd.notna(upload), None)

upload.to_sql("complaints", con=engine, if_exists="append", index=False, method="multi", chunksize=500)
print("Upload complete.")

Upload complete.


### Create a view for analysis

In [15]:
view_sql = (
    "CREATE OR REPLACE VIEW vw_complaint_details AS "
    "SELECT c.complaint_id, c.created_date, c.closed_date, "
    "a.agency_code, a.agency_name, cc.complaint_type, cc.descriptor, "
    "c.status, c.borough, c.incident_zip, "
    "CASE WHEN c.closed_date IS NOT NULL AND c.closed_date >= c.created_date "
    "THEN TIMESTAMPDIFF(SECOND, c.created_date, c.closed_date) / 3600.0 "
    "ELSE NULL END AS resolution_hours "
    "FROM complaints c "
    "JOIN agencies a ON c.agency_id = a.agency_id "
    "JOIN complaint_categories cc ON c.category_id = cc.category_id"
)
with engine.begin() as connection:
    connection.execute(text(view_sql))

print("View created.")

View created.


In [17]:
for table_name in ["agencies", "complaint_categories", "complaints"]:
    result = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table_name}", con=engine)
    print(f"{table_name}: {int(result.loc[0, 'row_count'])} rows")

display(pd.read_sql("SELECT * FROM vw_complaint_details LIMIT 10", con=engine))

agencies: 9 rows
complaint_categories: 55 rows
complaints: 300 rows


,complaint_id,created_date,closed_date,agency_code,agency_name,complaint_type,descriptor,status,borough,incident_zip,resolution_hours
0,63945513,2025-01-31 23:19:03,2025-02-21 16:12:41,DCWP,Department of Consumer and Worker Protection,Consumer Complaint,Restaurant,Closed,BROOKLYN,11234.0,496.8939
1,63941412,2025-01-31 23:35:00,2025-02-03 20:45:00,DEP,Department of Environmental Protection,Noise,Noise: Construction Before/After Hours (NM1),Closed,QUEENS,11372.0,69.1667
2,63943943,2025-01-31 23:53:00,2025-02-04 18:45:00,DEP,Department of Environmental Protection,Noise,Noise: Construction Before/After Hours (NM1),Closed,QUEENS,11428.0,90.8667
3,63943975,2025-01-31 23:32:00,2025-02-01 01:15:00,DEP,Department of Environmental Protection,Water System,Leak (Use Comments) (WA2),Closed,QUEENS,11364.0,1.7167
4,63945669,2025-01-31 23:44:00,2025-02-03 17:24:00,DEP,Department of Environmental Protection,Noise,Noise: air condition/ventilation equipment (NV1),Closed,MANHATTAN,10011.0,65.6667
5,63946136,2025-01-31 23:29:00,2025-02-01 01:00:00,DEP,Department of Environmental Protection,Noise,Noise: Construction Before/After Hours (NM1),Closed,STATEN ISLAND,10302.0,1.5167
6,63946383,2025-01-31 23:56:00,2025-02-03 10:03:00,DEP,Department of Environmental Protection,Noise,Noise: Alarms (NR3),Closed,QUEENS,11369.0,58.1167
7,63947645,2025-01-31 23:47:00,2025-02-01 01:25:00,DEP,Department of Environmental Protection,Sewer,Sewer Backup (Use Comments) (SA),Closed,QUEENS,11419.0,1.6333
8,63948067,2025-01-31 23:29:00,2025-02-03 16:23:00,DEP,Department of Environmental Protection,Noise,Noise: air condition/ventilation equipment (NV1),Closed,MANHATTAN,10009.0,64.9000
9,63949928,2025-01-31 23:17:00,2025-02-05 09:30:00,DEP,Department of Environmental Protection,Air Quality,"Air: Odor/Fumes, Restaurant (AD2)",Closed,MANHATTAN,10002.0,106.2167
